# TP2 — Part D: frozen PLM as a feature extractor (+ everything that depends on it)

| TP item | Where |
|---|---|
| **D1** `arabertv02-twitter` frozen → [CLS] vs attention-mask-weighted mean → logistic regression, threshold on val | §2 |
| D1 extras that the questions ask for: last layer vs last-4 layers, preprocessing ablation (Q3), imbalance handling (Q6), MARBERT frozen for "AraBERT vs MARBERT" (Q8) | §2.2 – §2.5 |
| **D2** all token states of the frozen encoder → BiLSTM + attention head (the C2 head), 3 seeds, **attention on 5 sarcastic tweets** (+ do attention weights sit on emoji / laughter / punctuation? Q7) | §3 |
| Final models: test scores, `predictions/*.csv`, CPU latency, params, logged to `results/results.csv` | §4 |
| **F1–F6 applied to the D models** (prior shift & F1=f(threshold), unseen dialect, shortcuts URL/mention, cue ablation, bootstrap CI + McNemar, error analysis) | §5 |
| Q8 table (frozen vs fine-tuned, cost/gain), Q10, Q11 inputs | §6 |

**Protocol:** all choices (preprocessing, pooling, layers, `C`, head hyper-parameters, epochs, threshold) are made on `val`.
Test is first touched in §4 for the *retained* D models only; §5 is analysis on test (never used to select anything).
Everything expensive is cached in `results/cache/` — re-running the notebook is cheap.

Needs: `pip install transformers torch scikit-learn emoji seaborn` · optional `pip install arabert arabic-reshaper python-bidi`
(AraBERT preprocessor, and correct Arabic glyph order in the attention figure). A T4 GPU is enough.

In [ ]:
import os, sys, re, time, copy, html, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd, torch, torch.nn as nn, emoji
import matplotlib.pyplot as plt, seaborn as sns
from IPython.display import HTML, display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.pipeline import make_pipeline, make_union
from sklearn.preprocessing import StandardScaler

from src.data import get_data
from src.evaluate import (PREDICTIONS, RESULTS, bootstrap_ci, f1_by_threshold, latency_per_tweet, log_result,
                          mcnemar, metrics, save_predictions)
from src.preprocess import build_pipeline
from src.train import best_threshold, count_params, fit_with_threshold, oversample, set_seed
from src.plm import (cached_npz, choose_max_length, extract_pooled, extract_tokens, get_device, length_stats,
                     load_encoder, plot_lengths, token_lengths, truncation_table)
from transformers import AutoTokenizer
from src.models import BiLSTMAttention
from src import analysis as A

SEED, SEEDS = 42, [42, 43, 44]
set_seed(SEED)
DEVICE = get_device()
CACHE, FIG = ROOT / "results" / "cache", ROOT / "results" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

MODEL_AT = "aubmindlab/bert-base-arabertv02-twitter"
MODEL_MB = "UBC-NLP/MARBERTv2"
RUN_MARBERT = True          # frozen MARBERT for the AraBERT-vs-MARBERT question (Q8)
RUN_D2_RETRAINS = True      # F2/F3 retrainings for the D2 head too (a few extra minutes)
BASELINE_NAME = "tfidf_lr"  # name of your baseline predictions file: predictions/<name>_seed42.csv
print("device:", DEVICE)

In [ ]:
train, val, test, report = get_data()
ytr, yva, yte = (d["sarcasm"].values.astype(int) for d in (train, val, test))
SPLITS = {"train": train, "val": val, "test": test}
print(f"train={len(train)}  val={len(val)}  test={len(test)}")

def by_dialect(df):
    return df.groupby("dialect")["sarcasm"].agg(n="size", sarcasm_rate="mean")
dialects = pd.concat({"train": by_dialect(train), "val": by_dialect(val), "test": by_dialect(test)}, axis=1)
display(dialects.round(3))
print(f"Q11 input — magreb tweets: train={int((train.dialect=='magreb').sum())}  val={int((val.dialect=='magreb').sum())}  test={int((test.dialect=='magreb').sum())}")
print(f"Q10 input — sarcasm rate: val={yva.mean():.3f}  test={yte.mean():.3f}")

## 1. Preprocessing variants for a PLM and encoder loading

In [ ]:
# PLM preprocessing = keep the sarcasm cues (emoji, elongation, punctuation, laughter); never stem / drop diacritics for a PLM.
PREPS = {
    "raw": lambda t: t,
    "base": build_pipeline(["html", "mentions", "hashtags"]),
}
try:
    from arabert.preprocess import ArabertPreprocessor
    try:
        _ap = ArabertPreprocessor(model_name=MODEL_AT, keep_emojis=True)
    except TypeError:
        _ap = ArabertPreprocessor(model_name=MODEL_AT)
    PREPS["arabert"] = _ap.preprocess
except Exception as e:
    print("AraBERT preprocessor not available (pip install arabert) -> skipped:", repr(e)[:100])
for n, f in PREPS.items():
    print(f"{n:<8}", f(val.tweet.iloc[5])[:100])

ENC = {}
def get_encoder(name):
    if name not in ENC:
        ENC[name] = load_encoder(name)
    return ENC[name]

tok_at, enc_at = get_encoder(MODEL_AT)
N_ENC_PARAMS = sum(p.numel() for p in enc_at.parameters())
print(f"encoder params: {N_ENC_PARAMS/1e6:.1f}M (all frozen)")

### 1.1 Choosing the maximum sequence length from the data
Rule (decided on **train only**): measure the token-length distribution for every tokenizer × preprocessing, look at the
mean / median / Q3 / Tukey fence (Q3 + 1.5·IQR) / p99 / p99.5 / max, apply `LENGTH_RULE`, round up to a multiple of 8,
and take the largest cap over the tokenizer × preprocessing combinations so nobody is truncated more than intended.
`MAX_LEN` is used for pooled features (D1; free, since batches are padded dynamically);
`TOK_MAX = min(MAX_LEN, RAM_CAP_TOK)` is used for D2 where the fixed-size array costs RAM.
Why not mean ± k·std: lengths are right-skewed, so a mean-based cut-off is unreliable; percentiles / the Tukey fence make no symmetry assumption.

In [ ]:
LENGTH_RULE = "p99.5"      # "tukey" | "p99" | "p99.5" | "mean+3std" | "max"   <- change here and re-run this cell
RAM_CAP_TOK = 128          # hard cap for the stored token states of D2 (RAM)

_prep_texts = {}
def prep_texts(prep, split):
    if (prep, split) not in _prep_texts:
        _prep_texts[(prep, split)] = [PREPS[prep](t) for t in SPLITS[split]["tweet"]]
    return _prep_texts[(prep, split)]

tokenizers = {"arabert-twitter": tok_at}
if RUN_MARBERT:
    tokenizers["marbert"] = AutoTokenizer.from_pretrained(MODEL_MB)

LEN = {}                       # (tokenizer, prep) -> {split: lengths}
for tn, tk in tokenizers.items():
    for pn in PREPS:
        LEN[(tn, pn)] = {s: token_lengths(tk, prep_texts(pn, s), 512) for s in SPLITS}

stats = pd.DataFrame({f"{tn} | {pn}": length_stats(v["train"]) for (tn, pn), v in LEN.items()}).T
caps = {f"{tn} | {pn}": choose_max_length(length_stats(v["train"]), LENGTH_RULE) for (tn, pn), v in LEN.items()}
display(stats.round(1).assign(cap_for_rule=pd.Series(caps)))

MAX_LEN = max(caps.values())
TOK_MAX = min(MAX_LEN, RAM_CAP_TOK)
print(f"rule = {LENGTH_RULE} -> MAX_LEN = {MAX_LEN} (pooled features) | TOK_MAX = {TOK_MAX} (D2 token states)")
print(f"D2 RAM for train token states ≈ {len(train) * TOK_MAX * enc_at.config.hidden_size * 2 / 1e9:.2f} GB (fp16)")

plot_lengths({k: v["train"] for k, v in LEN.items()}, chosen={k: MAX_LEN for k in LEN},
             path=FIG / "token_length_distribution.png")

# what every candidate rule would truncate, per split (check that val/test behave like train)
ref = ("arabert-twitter", "base" if "base" in PREPS else list(PREPS)[0])
cand = {r: choose_max_length(length_stats(LEN[ref]["train"]), r) for r in ["tukey", "p99", "p99.5", "mean+3std", "max"]}
cand["CHOSEN MAX_LEN"], cand["CHOSEN TOK_MAX"] = MAX_LEN, TOK_MAX
print(f"share of tweets truncated by each candidate cap ({ref[0]} | {ref[1]}):")
display((truncation_table(LEN[ref], cand) * 100).round(2).rename_axis("cap").rename(columns=lambda c: c + " (%)"))

## 2. D1 — frozen encoder → pooled vector → logistic regression

Pooling compared: `cls` vs attention-mask-weighted `mean`, from the **last layer** or the **average of the last 4 layers**,
and their concatenation. Features are standardised (scaler fitted on train inside the pipeline), LR uses `class_weight="balanced"`,
the threshold maximises sarcastic F1 on val. One forward pass over the data gives all pooling variants; the result is cached.

In [ ]:
def pooled(model_name, prep, split):
    texts = prep_texts(prep, split)
    path = CACHE / f"d_pooled_{model_name.split('/')[-1]}_{prep}_{split}_{len(texts)}_{MAX_LEN}.npz"
    def run():
        tok, enc = get_encoder(model_name)
        return extract_pooled(texts, tok, enc, MAX_LEN)
    return cached_npz(path, run)

FEATS = ["cls_last", "mean_last", "cls_last4", "mean_last4", "cls_last+mean_last"]
def feats(d, kind):
    return np.hstack([d[k] for k in kind.split("+")])

C_GRID = [0.01, 0.1, 1.0]
def fit_lr(Xtr, y_tr, Xva, y_va, C, class_weight="balanced"):
    model = make_pipeline(StandardScaler(), LogisticRegression(C=C, class_weight=class_weight, max_iter=3000))
    thr, secs = fit_with_threshold(model, Xtr, y_tr, Xva, y_va)
    return model, thr, secs

def run_d1(model_name, short, preps, feat_list):
    rows = []
    for prep in preps:
        dtr, dva = pooled(model_name, prep, "train"), pooled(model_name, prep, "val")
        ext = float(dtr["seconds"]) + float(dva["seconds"])
        for f in feat_list:
            Xtr, Xva = feats(dtr, f), feats(dva, f)
            for C in C_GRID:
                model, thr, secs = fit_lr(Xtr, ytr, Xva, yva, C)
                m = metrics(yva, model.predict_proba(Xva)[:, 1], thr)
                rows.append({"encoder": short, "prep": prep, "feats": f, "C": C, "threshold": thr,
                             "fit_time": secs, "extract_time": ext, "n_feat": Xtr.shape[1], **m})
    return pd.DataFrame(rows)

grid_at = run_d1(MODEL_AT, "arabert_tw", list(PREPS), FEATS)
best_cfg = grid_at.sort_values("f1", ascending=False).groupby(["prep", "feats"]).head(1)
display(best_cfg.pivot(index="feats", columns="prep", values="f1").round(4).rename_axis(columns="val F1 (best C)"))
display(best_cfg.pivot(index="feats", columns="prep", values="pr_auc").round(4).rename_axis(columns="val PR-AUC (best C)"))

### 2.1 Retained D1 configuration (selected on val) and logging

In [ ]:
def log_grid(best_rows, encoder_params):
    for _, r in best_rows.iterrows():
        log_result(model=f"d1_{r.encoder}_frozen_{r.feats}_lr", preprocessing=r.prep, seed=SEED, threshold=r.threshold,
                   train_time=r.extract_time + r.fit_time, extract_time=r.extract_time, fit_time=r.fit_time,
                   trained_params=int(r.n_feat) + 1, total_params=encoder_params + int(r.n_feat) + 1, C=r.C,
                   **{f"val_{k}": r[k] for k in ["f1", "pr_auc", "macro_f1", "precision", "recall"]})
log_grid(best_cfg, N_ENC_PARAMS)

top = grid_at.sort_values("f1", ascending=False).iloc[0]
BEST_PREP, BEST_F, BEST_C = top.prep, top.feats, float(top.C)
print(f"D1 retained: prep={BEST_PREP} | features={BEST_F} | C={BEST_C} | val F1={top.f1:.4f} PR-AUC={top.pr_auc:.4f} thr={top.threshold:.3f}")
print("\n[CLS] vs mean (best over layers/C), per preprocessing:")
display(best_cfg.assign(pool=best_cfg.feats.str.split("_").str[0]).groupby(["prep", "pool"])["f1"].max().unstack().round(4))

### 2.2 Imbalance handling on the frozen features (cheap version of Q6: weights vs over-sampling vs threshold)

In [ ]:
dtr, dva = pooled(MODEL_AT, BEST_PREP, "train"), pooled(MODEL_AT, BEST_PREP, "val")
Xtr, Xva = feats(dtr, BEST_F), feats(dva, BEST_F)
pos = oversample(train.assign(pos=np.arange(len(train))), seed=SEED)["pos"].values      # train only, after the split

def variant(name, X, y, cw):
    model, thr, secs = fit_lr(X, y, Xva, yva, BEST_C, class_weight=cw)
    proba = model.predict_proba(Xva)[:, 1]
    return {"strategy": name, "F1 @0.5": metrics(yva, proba, 0.5)["f1"], "F1 @val-opt thr": metrics(yva, proba, thr)["f1"],
            "thr": thr, "PR-AUC": metrics(yva, proba, thr)["pr_auc"], "fit_s": secs}
imb = pd.DataFrame([
    variant("(a) weighted loss + thr on val", Xtr, ytr, "balanced"),
    variant("(b) over-sampling (train only) + thr", Xtr[pos], ytr[pos], None),
    variant("(c) standard loss + thr on val", Xtr, ytr, None),
]).set_index("strategy")
display(imb.round(4))

### 2.3 AraBERT-twitter vs MARBERTv2, both frozen (Q8)

In [ ]:
if RUN_MARBERT:
    grid_mb = run_d1(MODEL_MB, "marbert", ["raw", "base"], ["cls_last", "mean_last", "mean_last4", "cls_last+mean_last"])
    N_MB_PARAMS = sum(p.numel() for p in get_encoder(MODEL_MB)[1].parameters())
    best_mb = grid_mb.sort_values("f1", ascending=False).groupby(["prep", "feats"]).head(1)
    log_grid(best_mb, N_MB_PARAMS)
    cmp = pd.concat([grid_at[grid_at.prep.isin(["raw", "base"])], grid_mb]).sort_values("f1", ascending=False)
    cmp = cmp.groupby(["encoder", "feats"]).head(1).pivot(index="feats", columns="encoder", values="f1")
    display(cmp.round(4).rename_axis(columns="val F1, frozen + LR"))
    print("best frozen MARBERT: F1 =", round(grid_mb.f1.max(), 4), "| best frozen AraBERT-twitter: F1 =", round(grid_at.f1.max(), 4))
else:
    print("RUN_MARBERT=False")

## 3. D2 — all token states of the frozen encoder → BiLSTM + attention pooling (head of C2)

Token states are extracted **once** (fp16, cached) with the retained preprocessing; only the small head is trained
(weighted CE, AdamW, early stopping on val sarcastic-F1 with the threshold re-optimised each epoch, 3 seeds).
The pooling is `α_i = softmax(vᵀ tanh(W h_i))`, `c = Σ α_i h_i` (in `src/models.py`, padding masked).

In [ ]:
def tokens(model_name, prep, split):
    texts = prep_texts(prep, split)
    path = CACHE / f"d_tokens_{model_name.split('/')[-1]}_{prep}_{split}_{len(texts)}_{TOK_MAX}.npz"
    def run():
        tok, enc = get_encoder(model_name)
        return extract_tokens(texts, tok, enc, TOK_MAX)
    return cached_npz(path, run)

T = {s: tokens(MODEL_AT, BEST_PREP, s) for s in ("train", "val")}
print({s: (v["states"].shape, f"{v['states'].nbytes/1e9:.2f} GB") for s, v in T.items()})

HP = dict(hidden=128, dropout=0.3, lr=1e-3, batch=64, epochs=20, patience=4, weight_decay=0.01)

def head_proba(model, S, M, idx=None, bs=256):
    dev = next(model.parameters()).device
    idx = np.arange(len(S)) if idx is None else idx
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(idx), bs):
            j = idx[i:i + bs]
            x = torch.from_numpy(S[j]).to(dev).float(); m = torch.from_numpy(M[j]).to(dev)
            out.append(torch.softmax(model(x, m), -1)[:, 1].cpu().numpy())
    return np.concatenate(out)

def train_head(tr, y_tr, va, y_va, seed, idx_tr=None, idx_va=None, hp=HP, verbose=True):
    set_seed(seed); rng = np.random.default_rng(seed)
    idx_tr = np.arange(len(y_tr)) if idx_tr is None else idx_tr
    idx_va = np.arange(len(y_va)) if idx_va is None else idx_va
    yv = y_va[idx_va]
    model = BiLSTMAttention(tr["states"].shape[-1], hp["hidden"], 2, hp["dropout"]).to(DEVICE)
    counts = np.bincount(y_tr[idx_tr], minlength=2)
    weights = torch.tensor(len(idx_tr) / (2 * counts), dtype=torch.float32, device=DEVICE)   # weighted loss
    loss_fn = nn.CrossEntropyLoss(weight=weights)
    opt = torch.optim.AdamW(model.parameters(), lr=hp["lr"], weight_decay=hp["weight_decay"])
    best, bad, hist, start = {"f1": -1}, 0, [], time.time()
    for ep in range(hp["epochs"]):
        model.train(); perm = rng.permutation(idx_tr); total = 0.0
        for i in range(0, len(perm), hp["batch"]):
            j = perm[i:i + hp["batch"]]
            x = torch.from_numpy(tr["states"][j]).to(DEVICE).float(); m = torch.from_numpy(tr["masks"][j]).to(DEVICE)
            y = torch.from_numpy(y_tr[j]).long().to(DEVICE)
            loss = loss_fn(model(x, m), y)
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            total += loss.item() * len(j)
        proba = head_proba(model, va["states"], va["masks"], idx_va)
        thr = best_threshold(yv, proba); f1 = f1_score(yv, proba >= thr, zero_division=0)
        hist.append({"epoch": ep + 1, "train_loss": total / len(perm), "val_f1": f1, "thr": thr})
        if f1 > best["f1"]:
            best = {"f1": f1, "thr": thr, "epoch": ep + 1, "state": copy.deepcopy(model.state_dict())}; bad = 0
        else:
            bad += 1
            if bad >= hp["patience"]:
                break
    model.load_state_dict(best["state"])
    if verbose:
        print(f"seed {seed}: best epoch {best['epoch']}/{len(hist)} | val F1 {best['f1']:.4f} | thr {best['thr']:.3f} | {time.time()-start:.0f}s")
    return model, best["thr"], time.time() - start, pd.DataFrame(hist)

D2 = {}
for s in SEEDS:
    model, thr, secs, hist = train_head(T["train"], ytr, T["val"], yva, s)
    proba = head_proba(model, T["val"]["states"], T["val"]["masks"])
    D2[s] = {"model": model, "thr": thr, "time": secs, "hist": hist, "val": metrics(yva, proba, thr), "val_proba": proba}
ext_time = sum(float(T[s]["seconds"]) for s in T)
d2_val = pd.DataFrame({s: v["val"] for s, v in D2.items()}).T
display(d2_val.round(4))
print("val F1 mean ± std over 3 seeds:", f"{d2_val.f1.mean():.4f} ± {d2_val.f1.std():.4f}",
      "| PR-AUC:", f"{d2_val.pr_auc.mean():.4f} ± {d2_val.pr_auc.std():.4f}")
N_HEAD = count_params(D2[SEED]["model"])
for s, v in D2.items():
    log_result(model="d2_arabert_frozen_bilstm_attn", preprocessing=BEST_PREP, seed=s, threshold=v["thr"],
               train_time=ext_time + v["time"], extract_time=ext_time, fit_time=v["time"], trained_params=N_HEAD,
               total_params=N_ENC_PARAMS + N_HEAD, **{f"val_{k}": x for k, x in v["val"].items()})
print(f"trained params (head only) = {N_HEAD:,} vs encoder (frozen) = {N_ENC_PARAMS:,}")

### 3.1 Attention on 5 sarcastic tweets (figure for the report) — D2

In [ ]:
try:
    import arabic_reshaper
    from bidi.algorithm import get_display
    shape = lambda s: get_display(arabic_reshaper.reshape(s))
except Exception:
    shape = lambda s: s
    print("tip: pip install arabic-reshaper python-bidi for correctly shaped Arabic labels")

def attention_for(model, S, M, idx):
    model.eval(); dev = next(model.parameters()).device
    x = torch.from_numpy(S[idx]).to(dev).float(); m = torch.from_numpy(M[idx]).to(dev)
    with torch.no_grad():
        logits, alpha = model(x, m, return_attention=True)
    return torch.softmax(logits, -1)[:, 1].cpu().numpy(), alpha.cpu().numpy()

m42, thr42 = D2[SEED]["model"], D2[SEED]["thr"]
p42 = D2[SEED]["val_proba"]
ok = np.where((yva == 1) & (p42 >= thr42))[0]
# prefer tweets that actually contain a cue so the figure is informative; fall back to random ones
has_cue = np.array([A.CUES["all_cues"][0](val.tweet.iloc[i]) for i in ok])
rng = np.random.default_rng(7)
PICK = list(rng.choice(ok[has_cue], 3, replace=False)) + list(rng.choice(ok[~has_cue], 2, replace=False)) if (has_cue.sum() >= 3 and (~has_cue).sum() >= 2) else list(rng.choice(ok, 5, replace=False))
probs, alphas = attention_for(m42, T["val"]["states"], T["val"]["masks"], np.array(PICK))

fig, axes = plt.subplots(5, 1, figsize=(11, 15))
for ax, i, pr, al in zip(axes, PICK, probs, alphas):
    n = int(T["val"]["masks"][i].sum())
    toks = tok_at.convert_ids_to_tokens(T["val"]["ids"][i][:n])
    ax.bar(range(n), al[:n], color="#4C78A8")
    ax.set_xticks(range(n)); ax.set_xticklabels([shape(t) for t in toks], rotation=90, fontsize=9)
    ax.set_title(f"val tweet #{i} — P(sarcastic)={pr:.2f}  (tokens in logical order)", fontsize=10)
    ax.set_ylabel("attention α")
plt.tight_layout(); plt.savefig(FIG / "d2_attention_5_tweets.png", dpi=200); plt.show()

for i, al in zip(PICK, alphas):
    n = int(T["val"]["masks"][i].sum()); toks = tok_at.convert_ids_to_tokens(T["val"]["ids"][i][:n])
    mx = al[:n].max()
    spans = "".join(f'<span style="background:rgba(255,90,0,{a/mx:.2f});padding:2px 3px;margin:1px;border-radius:3px">{html.escape(t)}</span>' for t, a in zip(toks, al[:n]))
    display(HTML(f'<div dir="rtl" style="font-size:19px;line-height:2.3;border-bottom:1px solid #ddd">{spans}</div>'))

### 3.2 Does the attention sit on the expected markers? (relative attention = α·(#tokens); 1 = uniform)

In [ ]:
SPECIAL = set(tok_at.all_special_tokens)
def token_kind(t):
    if t in SPECIAL: return "[CLS]/[SEP]"
    s = t.replace("##", "")
    if emoji.emoji_count(s): return "emoji"
    if re.fullmatch(r"(?:ها|ه){2,}|خ{2,}", s): return "laughter"
    if re.search(r"(.)\1{2,}", s): return "elongation"
    if re.fullmatch(r"[^\w\s]+", s): return "punctuation"
    return "word"

det = np.where((yva == 1) & (p42 >= thr42))[0]
rel = []
for b in range(0, len(det), 256):
    idx = det[b:b + 256]
    _, al = attention_for(m42, T["val"]["states"], T["val"]["masks"], idx)
    for k, i in enumerate(idx):
        n = int(T["val"]["masks"][i].sum())
        for t, a in zip(tok_at.convert_ids_to_tokens(T["val"]["ids"][i][:n]), al[k][:n]):
            rel.append((token_kind(t), a * n))
rel = pd.DataFrame(rel, columns=["kind", "relative_attention"])
att_table = rel.groupby("kind")["relative_attention"].agg(["mean", "count"]).sort_values("mean", ascending=False)
display(att_table.round(3))

## 4. Final D models: test scores, predictions, CPU latency, logging
Only the **retained** models touch the test set here: D1 (`BEST_PREP / BEST_F / BEST_C`) and D2 (3 seeds).

In [ ]:
# ---- D1 final (refit = same as grid row, deterministic)
D1_model, D1_thr, D1_fit = fit_lr(Xtr, ytr, Xva, yva, BEST_C)
dte = pooled(MODEL_AT, BEST_PREP, "test")
Xte = feats(dte, BEST_F)
D1_val = metrics(yva, D1_model.predict_proba(Xva)[:, 1], D1_thr)
D1_test_proba = D1_model.predict_proba(Xte)[:, 1]
D1_test = metrics(yte, D1_test_proba, D1_thr)
D1_NAME = f"d1_arabert_tw_frozen_{BEST_F}_lr"      # same string as the logged model name
save_predictions(D1_NAME, SEED, test["id"].values, yte, D1_test_proba, D1_thr)

def d1_predict(texts, tok=None, enc=None, bs=64, prep=None):
    tok, enc = (tok, enc) if enc is not None else get_encoder(MODEL_AT)
    d = extract_pooled([(prep or PREPS[BEST_PREP])(x) for x in texts], tok, enc, MAX_LEN, bs)
    return D1_model.predict_proba(feats(d, BEST_F))[:, 1]

# ---- D2 final (3 seeds)
T["test"] = tokens(MODEL_AT, BEST_PREP, "test")
D2_NAME = "d2_arabert_frozen_bilstm_attn"
for s, v in D2.items():
    v["test_proba"] = head_proba(v["model"], T["test"]["states"], T["test"]["masks"])
    v["test"] = metrics(yte, v["test_proba"], v["thr"])
    save_predictions(D2_NAME, s, test["id"].values, yte, v["test_proba"], v["thr"])

def d2_predict(texts, seed=SEED, tok=None, enc=None, bs=64, prep=None, head=None):
    tok, enc = (tok, enc) if enc is not None else get_encoder(MODEL_AT)
    d = extract_tokens([(prep or PREPS[BEST_PREP])(x) for x in texts], tok, enc, TOK_MAX, bs)
    return head_proba(head or D2[seed]["model"], d["states"], d["masks"])

# ---- CPU latency, batch of 1 (what a production endpoint would see)
enc_cpu = copy.deepcopy(enc_at).float().cpu().eval()
head_cpu = copy.deepcopy(D2[SEED]["model"]).cpu()
sample = val.tweet.tolist()[:50]
LAT_D1 = latency_per_tweet(lambda tx: [d1_predict([t], tok_at, enc_cpu, 1) for t in tx], sample)
LAT_D2 = latency_per_tweet(lambda tx: [d2_predict([t], SEED, tok_at, enc_cpu, 1, head=head_cpu) for t in tx], sample)
print(f"CPU latency / tweet: D1 = {LAT_D1*1000:.0f} ms | D2 = {LAT_D2*1000:.0f} ms")
del enc_cpu, head_cpu

# ---- logging (overwrites the val-only rows with val + test)
log_result(model=f"d1_arabert_tw_frozen_{BEST_F}_lr", preprocessing=BEST_PREP, seed=SEED, threshold=D1_thr,
           train_time=float(dtr["seconds"]) + float(dva["seconds"]) + D1_fit, trained_params=Xtr.shape[1] + 1,
           total_params=N_ENC_PARAMS + Xtr.shape[1] + 1, C=BEST_C, latency_cpu_s=LAT_D1,
           **{f"val_{k}": v for k, v in D1_val.items()}, **{f"test_{k}": v for k, v in D1_test.items()})
for s, v in D2.items():
    log_result(model=D2_NAME, preprocessing=BEST_PREP, seed=s, threshold=v["thr"], train_time=ext_time + v["time"],
               trained_params=N_HEAD, total_params=N_ENC_PARAMS + N_HEAD, latency_cpu_s=LAT_D2,
               **{f"val_{k}": x for k, x in v["val"].items()}, **{f"test_{k}": x for k, x in v["test"].items()})

final = pd.DataFrame({"D1 (LR)": {**{f"val_{k}": D1_val[k] for k in ("f1", "pr_auc")}, **{f"test_{k}": D1_test[k] for k in ("f1", "pr_auc", "macro_f1")}},
                      "D2 mean": {**{f"val_{k}": np.mean([v['val'][k] for v in D2.values()]) for k in ("f1", "pr_auc")},
                                  **{f"test_{k}": np.mean([v['test'][k] for v in D2.values()]) for k in ("f1", "pr_auc", "macro_f1")}},
                      "D2 std": {**{f"val_{k}": np.std([v['val'][k] for v in D2.values()], ddof=1) for k in ("f1", "pr_auc")},
                                 **{f"test_{k}": np.std([v['test'][k] for v in D2.values()], ddof=1) for k in ("f1", "pr_auc", "macro_f1")}}})
display(final.round(4))
BEST_D = "D1" if D1_val["f1"] >= np.mean([v["val"]["f1"] for v in D2.values()]) else "D2"
print("Best D model according to val:", BEST_D)

## 5. Part F applied to the D models (analysis on test — never used for selection)

In [ ]:
# ---- baseline predictions (created here only if your baseline notebook did not save them)
def ensure_baseline():
    if A.load_predictions(BASELINE_NAME, SEED) is not None:
        return
    print("baseline predictions not found -> training the best TF-IDF baseline (emoji + norm + light_stem)")
    prep = build_pipeline(["html", "mentions", "hashtags", "emoji", "diacritics", "kashida", "letters", "elongation", "light_stem"])
    Xs = {s: SPLITS[s]["tweet"].apply(prep) for s in SPLITS}
    model = make_pipeline(make_union(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
                                     TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)),
                          LogisticRegression(C=1.0, class_weight="balanced", max_iter=1000))
    thr, secs = fit_with_threshold(model, Xs["train"], ytr, Xs["val"], yva)
    mv, mt = metrics(yva, model.predict_proba(Xs["val"])[:, 1], thr), metrics(yte, model.predict_proba(Xs["test"])[:, 1], thr)
    save_predictions(BASELINE_NAME, SEED, test["id"].values, yte, model.predict_proba(Xs["test"])[:, 1], thr)
    log_result(model=BASELINE_NAME, preprocessing="emoji + norm + light_stem", seed=SEED, threshold=thr, train_time=secs,
               **{f"val_{k}": v for k, v in mv.items()}, **{f"test_{k}": v for k, v in mt.items()})
ensure_baseline()

def logged_threshold(model, seed=SEED):
    r = pd.read_csv(RESULTS); r = r[(r.model == model) & (r.seed == seed)]
    return float(r.sort_values("val_f1", ascending=False).iloc[0]["threshold"])

meta = test[["id", "tweet", "dialect", "sentiment", "key"]]
def load(name, seed=SEED):
    return A.load_predictions(name, seed).merge(meta, on="id")
PRED = {"Baseline TF-IDF": load(BASELINE_NAME), "D1 frozen+LR": load(D1_NAME), "D2 frozen+BiLSTM-attn (seed 42)": load(D2_NAME)}
THR = {"Baseline TF-IDF": logged_threshold(BASELINE_NAME), "D1 frozen+LR": D1_thr, "D2 frozen+BiLSTM-attn (seed 42)": D2[SEED]["thr"]}

### F1 — prior shift: val vs test sarcasm rate, F1 at the val threshold, and F1 = f(threshold) on test (Q10)

In [ ]:
shift = pd.DataFrame({n: A.prior_shift(yva, yte, df["proba"].values, THR[n]) for n, df in PRED.items()}).T
display(shift.round(4))
curves = {n: f1_by_threshold(yte, df["proba"].values) for n, df in PRED.items()}
A.plot_f1_threshold(curves, THR, FIG / "f1_threshold_test_D.png")      # figure to paste in the report

### F2 — unseen dialect: retrain without `egypt`, evaluate on test `egypt` (D models are cheap to retrain)

In [ ]:
eg_tr, eg_va, eg_te = (train.dialect == "egypt").values, (val.dialect == "egypt").values, (test.dialect == "egypt").values
print(f"egypt tweets: train={eg_tr.sum()} ({eg_tr.mean():.1%})  val={eg_va.sum()}  test={eg_te.sum()} ({eg_te.mean():.1%})")

def f1_on(df, mask, proba=None, thr=None):
    y = df["y_true"].values[mask]
    p = df["y_pred"].values[mask] if proba is None else (proba[mask] >= thr).astype(int)
    return f1_score(y, p, zero_division=0)

rows = []
# D1: threshold chosen on val *without egypt* so no egypt information leaks
m_ne, thr_ne, _ = fit_lr(Xtr[~eg_tr], ytr[~eg_tr], Xva[~eg_va], yva[~eg_va], BEST_C)
p_ne = m_ne.predict_proba(Xte)[:, 1]
rows.append({"model": "D1", "F1 egypt (trained on all)": f1_on(PRED["D1 frozen+LR"], eg_te),
             "F1 egypt (egypt unseen)": f1_on(PRED["D1 frozen+LR"], eg_te, p_ne, thr_ne),
             "F1 non-egypt (all)": f1_on(PRED["D1 frozen+LR"], ~eg_te),
             "F1 non-egypt (egypt unseen)": f1_on(PRED["D1 frozen+LR"], ~eg_te, p_ne, thr_ne)})
if RUN_D2_RETRAINS:
    m2, thr2, _, _ = train_head(T["train"], ytr, T["val"], yva, SEED, np.where(~eg_tr)[0], np.where(~eg_va)[0])
    p2 = head_proba(m2, T["test"]["states"], T["test"]["masks"])
    d2df = PRED["D2 frozen+BiLSTM-attn (seed 42)"]
    rows.append({"model": "D2", "F1 egypt (trained on all)": f1_on(d2df, eg_te), "F1 egypt (egypt unseen)": f1_on(d2df, eg_te, p2, thr2),
                 "F1 non-egypt (all)": f1_on(d2df, ~eg_te), "F1 non-egypt (egypt unseen)": f1_on(d2df, ~eg_te, p2, thr2)})
f2 = pd.DataFrame(rows).set_index("model")
f2["drop on egypt"] = f2["F1 egypt (trained on all)"] - f2["F1 egypt (egypt unseen)"]
display(f2.round(4))

### F3 — shortcuts: with / without URL, mention, CSV-quote artefact; retrain without URL and mention tokens

In [ ]:
def subsets(df):
    t = df["tweet"]
    u, mn, q = t.map(A.has_url).values, t.map(A.has_mention).values, t.map(A.is_quoted).values
    return {"all": np.ones(len(df), bool), "with URL": u, "without URL": ~u, "with @mention": mn, "without @mention": ~mn,
            "quoted (CSV artefact)": q, "not quoted": ~q}
sub = pd.concat({n: A.subgroup_scores(df, subsets(df))[["n", "sarcasm_rate", "f1", "precision", "recall", "pr_auc"]] for n, df in PRED.items()})
display(sub.round(3))

# retrain without URL / mention tokens (removed from the raw text before the usual preprocessing)
NOURL = "nourl_" + BEST_PREP
PREPS[NOURL] = lambda t, b=PREPS[BEST_PREP]: b(A.remove_urls_mentions(t))
dtr2, dva2, dte2 = (pooled(MODEL_AT, NOURL, s) for s in ("train", "val", "test"))
m_nu, thr_nu, _ = fit_lr(feats(dtr2, BEST_F), ytr, feats(dva2, BEST_F), yva, BEST_C)
p_nu = m_nu.predict_proba(feats(dte2, BEST_F))[:, 1]
S1 = subsets(PRED["D1 frozen+LR"])
rows = [{"model": "D1 (original text)", "test F1": f1_on(PRED["D1 frozen+LR"], S1["all"]), "F1 with URL": f1_on(PRED["D1 frozen+LR"], S1["with URL"]),
         "F1 without URL": f1_on(PRED["D1 frozen+LR"], S1["without URL"]), "val F1": D1_val["f1"]},
        {"model": "D1 (URL+mention removed)", "test F1": f1_on(PRED["D1 frozen+LR"], S1["all"], p_nu, thr_nu),
         "F1 with URL": f1_on(PRED["D1 frozen+LR"], S1["with URL"], p_nu, thr_nu),
         "F1 without URL": f1_on(PRED["D1 frozen+LR"], S1["without URL"], p_nu, thr_nu),
         "val F1": metrics(yva, m_nu.predict_proba(feats(dva2, BEST_F))[:, 1], thr_nu)["f1"]}]
if RUN_D2_RETRAINS:
    T2 = {s: tokens(MODEL_AT, NOURL, s) for s in ("train", "val", "test")}
    m2n, thr2n, _, _ = train_head(T2["train"], ytr, T2["val"], yva, SEED)
    p2n = head_proba(m2n, T2["test"]["states"], T2["test"]["masks"])
    d2df = PRED["D2 frozen+BiLSTM-attn (seed 42)"]
    rows += [{"model": "D2 (original text)", "test F1": f1_on(d2df, S1["all"]), "F1 with URL": f1_on(d2df, S1["with URL"]),
              "F1 without URL": f1_on(d2df, S1["without URL"]), "val F1": D2[SEED]["val"]["f1"]},
             {"model": "D2 (URL+mention removed)", "test F1": f1_on(d2df, S1["all"], p2n, thr2n),
              "F1 with URL": f1_on(d2df, S1["with URL"], p2n, thr2n), "F1 without URL": f1_on(d2df, S1["without URL"], p2n, thr2n),
              "val F1": metrics(yva, head_proba(m2n, T2["val"]["states"], T2["val"]["masks"]), thr2n)["f1"]}]
    del T2
display(pd.DataFrame(rows).set_index("model").round(4))

### F4 — sarcasm cues: remove emojis / laughter / repeated punctuation on correctly detected sarcastic tweets

In [ ]:
f4 = {}
for name, fn, df, thr in [("D1", lambda tx: d1_predict(tx), PRED["D1 frozen+LR"], D1_thr),
                          ("D2", lambda tx: d2_predict(tx), PRED["D2 frozen+BiLSTM-attn (seed 42)"], D2[SEED]["thr"])]:
    f4[name] = A.cue_ablation(df["tweet"], df["y_true"], df["y_pred"], fn, thr, seed=SEED)
f4 = pd.concat(f4)
display(f4.round(3))
print("'drop' = share of previously detected sarcastic tweets (that contain the cue) no longer detected after removing it;")
print("'control_drop' = same with the same number of random words removed (is the cue special or is it just less text?).")

### F5 — significance: bootstrap CI (1000), 3 seeds, McNemar

In [ ]:
rows = []
for n, df in PRED.items():
    lo, hi = bootstrap_ci(df["y_true"], df["y_pred"], n=1000, seed=SEED)
    rows.append({"model": n, "test F1": f1_score(df["y_true"], df["y_pred"]), "CI95 low": lo, "CI95 high": hi})
display(pd.DataFrame(rows).set_index("model").round(4))
t2 = pd.DataFrame({s: v["test"] for s, v in D2.items()}).T
print(f"D2 test F1 over 3 seeds: {t2.f1.mean():.4f} ± {t2.f1.std():.4f} | PR-AUC {t2.pr_auc.mean():.4f} ± {t2.pr_auc.std():.4f}")
print("D1: deterministic (frozen encoder + convex LR) → no seed variance to report.")

names = list(PRED)
mc = pd.DataFrame([[mcnemar(PRED[a]["y_true"], PRED[a]["y_pred"], PRED[b]["y_pred"]) if a != b else np.nan for b in names] for a in names], index=names, columns=names)
display(mc.round(4).rename_axis(columns="McNemar p-value"))

# McNemar between the two best models *overall* (selected on val F1 among everything logged with predictions on disk)
res = pd.read_csv(RESULTS)
ranked = res.groupby("model")["val_f1"].mean().sort_values(ascending=False)
avail = [(m, res[res.model == m].seed.iloc[0]) for m in ranked.index if (PREDICTIONS / f"{m}_seed{int(res[res.model == m].seed.iloc[0])}.csv").exists()]
if len(avail) >= 2:
    (m1, s1), (m2_, s2) = avail[:2]
    a, b = A.load_predictions(m1, int(s1)), A.load_predictions(m2_, int(s2))
    print(f"two best on val with predictions: {m1} vs {m2_} → McNemar p = {mcnemar(a.y_true, a.y_pred, b.y_pred):.4f}")

### F6 — error analysis of the best D model (15 FN + 15 FP) and cross-check with contradictory-label duplicates

In [ ]:
best_name = "D1 frozen+LR" if BEST_D == "D1" else "D2 frozen+BiLSTM-attn (seed 42)"
dfe = PRED[best_name].copy()
conf = pd.read_csv(ROOT / "results" / "conflicts.csv")
dfe["in_conflict_group"] = dfe["key"].isin(set(conf["key"]))
dfe["has_emoji"], dfe["has_laugh"] = dfe.tweet.map(A.has_emoji), dfe.tweet.map(A.has_laugh)
dfe["has_repeat_punct"], dfe["has_url"] = dfe.tweet.map(A.has_repeat_punct), dfe.tweet.map(A.has_url)
errs = A.error_sample(dfe, 15, SEED)
errs["category"] = ""            # <- fill by hand in the CSV, one of: " + " | ".join(A.ERROR_CATEGORIES)
out = ROOT / "results" / f"error_analysis_{BEST_D}.csv"
errs[["id", "error", "tweet", "dialect", "sentiment", "proba", "in_conflict_group", "has_emoji", "has_laugh", "has_repeat_punct", "has_url", "category"]].to_csv(out, index=False)
print("model:", best_name, "| errors on test: FN =", int(((dfe.y_true == 1) & (dfe.y_pred == 0)).sum()), " FP =", int(((dfe.y_true == 0) & (dfe.y_pred == 1)).sum()))
print("saved for manual categorisation:", out, "| categories:", A.ERROR_CATEGORIES)
print(f"share of test tweets belonging to a contradictory-label group: {dfe.in_conflict_group.mean():.3%} | among all errors: {dfe[dfe.y_true != dfe.y_pred].in_conflict_group.mean():.3%} | among the 30 sampled: {errs.in_conflict_group.mean():.3%}")
print("dialect of all errors vs all test:"); display(pd.concat({"errors": dfe[dfe.y_true != dfe.y_pred].dialect.value_counts(normalize=True), "test": dfe.dialect.value_counts(normalize=True)}, axis=1).round(3))
pd.set_option("display.max_colwidth", 200)
display(errs[["error", "tweet", "dialect", "sentiment", "proba", "in_conflict_group"]].reset_index(drop=True))

In [ ]:
# run again after filling the `category` column of the CSV: counts for the report (Q12)
filled = pd.read_csv(ROOT / "results" / f"error_analysis_{BEST_D}.csv")
if filled["category"].notna().any():
    display(pd.crosstab(filled["category"].fillna("(unfilled)"), filled["error"], margins=True))
else:
    print("category column still empty — fill it in the CSV, then re-run this cell.")

## 6. Frozen vs fine-tuned, cost and gain (Q8) + figures of merit for Part G

In [ ]:
res = pd.read_csv(RESULTS)
def pick(prefixes):
    return res[res.model.str.startswith(tuple(prefixes))]
cols = [c for c in ["model", "preprocessing", "seed", "val_f1", "val_pr_auc", "test_f1", "test_pr_auc", "train_time",
                    "trained_params", "total_params", "latency_cpu_s"] if c in res.columns]
tab = pick([BASELINE_NAME, "b1_", "b2_", "d1_", "d2_", "e1_", "e2_"])[cols]
display(tab.sort_values("val_f1", ascending=False).round(4).reset_index(drop=True))
e1 = pick(["e1_"])
if len(e1):
    d, e = pick(["d1_", "d2_"]).val_f1.max(), e1.val_f1.max()
    print(f"Gain of fine-tuning over the best frozen D model on val F1: {e-d:+.4f}  (see train_time / trained_params columns for the cost)")
else:
    print("No E1 rows in results.csv yet — re-run this cell after Part E to fill the frozen-vs-fine-tuned comparison.")

**Éléments de réponse (à compléter avec vos chiffres) :**

* **Q8 — gelé vs fine-tuné.** Un encodeur gelé n'entraîne que `d+1` paramètres (D1) ou ≈ 0,5 M (tête D2) contre ≈ 135 M pour le fine-tuning, et s'entraîne en secondes une fois les représentations en cache, mais ses représentations n'ont pas été optimisées pour le sarcasme (pré-entraînement MLM/NSP) : l'écart de F1 val mesuré en §6 est le « gain » du fine-tuning, à mettre en regard du coût (temps, GPU, paramètres, 3 graines). Pooling : comparer [CLS] vs moyenne en §2.1 — le [CLS] d'un BERT non fine-tuné n'a été entraîné que pour la NSP (ou pas du tout pour AraBERTv02), la moyenne des tokens est en général meilleure ; la moyenne des 4 dernières couches ajoute de l'information moins spécialisée MLM.
* **AraBERT-twitter vs MARBERT.** §2.3 donne la comparaison à encodeur gelé ; la comparaison fine-tunée (E1) viendra de `results.csv`. Arguments : MARBERTv2 est pré-entraîné sur ≈ 1 Md de tweets arabes **multi-dialectaux** (donc plus proche du dialecte), AraBERTv02-twitter sur des tweets surtout MSA/dialectaux plus un corpus MSA ; leur vocabulaire et tokenizer diffèrent (emojis, élongations).
* **Q7 (D2).** §3.2 : « attention relative » > 1 pour emoji/rires/ponctuation = le pooling attentionnel se concentre sur ces marqueurs ; comparez avec les mots ordinaires et avec [CLS]/[SEP] (qui absorbent souvent beaucoup de masse). L'attention est une explication **partielle** (corrélation, pas causalité) : F4 mesure la causalité par ablation.
* **Q10.** F1 : comparer taux de sarcasme val/test, seuil val vs seuil optimal a posteriori et perte de F1 (table `shift`). En production : recalibration du seuil sur un échantillon récent annoté, suivi du taux de prédiction positive et de la distribution des scores (dérive de prior), alertes + ré-annotation périodique d'un échantillon.
* **Q11.** Nombre de tweets `magreb` : en tête de notebook. Pour le Maroc : constituer un petit jeu **annoté localement** (darija, arabizi, français mélangé), le garder sous scellés, estimer F1/PR-AUC avec IC bootstrap, fixer le seuil sur une moitié et évaluer sur l'autre ; comparer à la chute mesurée en F2 (Égypte non vue) comme ordre de grandeur de l'effet dialecte.